# Обучение (Colab, T4)

**Правило: в ноутбуке нет логики.** Весь код — в `src/`, ноутбук только запускает.

Перед запуском: Runtime → Change runtime type → **T4 GPU**.

В Drive (`MyDrive/scanq/`) кладём один файл — `scanq_dataset.tar`, 355 МБ. Собирается локально:

```powershell
python -m src.data.package --data data\raw\tobacco3482\data `
    --manifest data\synthetic\manifest.jsonl --labels data\labeled\labels.jsonl `
    --splits data\splits --out data\scanq_dataset.tar
```

Внутри: 537 страниц (251 эталон под синтетику + 300 размеченных вручную), рецепт
синтетики на 8000 страниц, разметка и три файла сплита.

Готовые картинки синтетики в Drive **не везём**: 8000 отрисованных страниц весят
9.4 ГБ, а рецепт — три мегабайта. Эталон, набор меток, силы дефектов и зерно
полностью задают страницу, поэтому здесь она соберётся побитово такой же.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/<user>/scan-quality.git /content/scan-quality
!cd /content/scan-quality && pip install -e ".[train]" -q

In [ ]:
# Распаковка в /content, а не работа прямо с Drive: чтение тысяч мелких файлов
# с Drive медленнее самого обучения.
!mkdir -p /content/data
!tar -xf /content/drive/MyDrive/scanq/scanq_dataset.tar -C /content/data
!ls /content/data && ls /content/data/pages | head -3

In [ ]:
# Синтетика собирается на месте по рецепту: эталон, метки, силы и зерно
# полностью задают страницу, поэтому результат побитово тот же, что локально.
!cd /content/scan-quality && python -m src.data.generate \
    --data /content/data/pages \
    --prelabels /content/data/manifest.jsonl \
    --splits /content/data/splits \
    --out /content/synthetic \
    --from-manifest /content/data/manifest.jsonl

In [ ]:
# Чекпоинт каждую эпоху на Drive + resume auto: сессия отвалится, это вопрос
# времени. Повторный запуск этой же ячейки продолжит с последней эпохи.
!cd /content/scan-quality && python -m src.models.train \
    --config configs/base.yaml \
    --data /content/data/pages \
    --synthetic /content/synthetic \
    --manifest /content/data/manifest.jsonl \
    --labels /content/data/labels.jsonl \
    --splits /content/data/splits \
    --out /content/drive/MyDrive/scanq/runs \
    --resume auto

In [ ]:
# Кривые обучения из CSV на Drive — он дописывается и переживает обрывы.
import pandas as pd

log = pd.read_csv('/content/drive/MyDrive/scanq/runs/metrics.csv')
display(log.tail(10))
log.plot(x='epoch', y=['train_loss', 'val_loss'], figsize=(9, 3))
log.plot(x='epoch', y=['macro_ap', 'macro_f1'], figsize=(9, 3))

In [ ]:
# Метрики по каждой метке на val. Тест не трогаем до С8.
!cd /content/scan-quality && python -m src.models.evaluate \
    --checkpoint /content/drive/MyDrive/scanq/runs/best.ckpt \
    --data /content/data/pages \
    --labels /content/data/labels.jsonl \
    --splits /content/data/splits \
    --part val --device cuda